# Day 2 - INSTRUCTOR Notebook: Passwords and Hashing
# Contains solutions and teaching notes. Do not share with students.

## Teaching note
Exercise 1 makes keyspace visceral. 26^8 = 208,827,064,576. At 1 billion
guesses/second that is about 209 seconds - under 4 minutes. Modern GPUs do
10-100 billion/second for simple hashes. 26^16 is 43 quadrillion, which is
about 500 days at GPU speed. Length wins.

In [ ]:
# Exercise 1 SOLUTION: Keyspace
N, L = 26, 8
keyspace = N ** L
print("keyspace:", keyspace)
rate = 1_000_000_000
print("seconds to crack:", keyspace / rate)

## Teaching note
Exercise 2 demonstrates the avalanche effect: one capital letter produces a
completely different hash. Stress that SHA-256 is one-way: you cannot reverse
it. Ask: if two people have the same password, do they have the same hash? Yes -
that is why salting matters (Exercise 3).

In [ ]:
# Exercise 2 SOLUTION: Hashing
import hashlib
h1 = hashlib.sha256(b"sunshine").hexdigest()
h2 = hashlib.sha256(b"Sunshine").hexdigest()
print("hash of sunshine:", h1)
print("hash of Sunshine:", h2)
print("same?", h1 == h2)  # False - completely different hashes

## Teaching note
Exercise 3: salts are stored in plaintext alongside the hash. Their job is to
make rainbow table attacks fail - an attacker cannot precompute hashes for all
common passwords if every hash has a unique salt mixed in. Run the cell twice
and show students the salt (and therefore hash) differs each time.

In [ ]:
# Exercise 3 SOLUTION: Salt
import secrets
salt = secrets.token_hex(8)  # 8 bytes = 16 hex chars
pw = "sunshine"
salted = (salt + pw).encode()
print("salt:", salt)
print("salted hash:", hashlib.sha256(salted).hexdigest())
# Run again - same password, different salt, different hash

## Pair work SOLUTION - Password strength checker

In [ ]:
# PAIR WORK SOLUTION
import hashlib

def check_password(pw):
    if len(pw) < 12:
        return False
    if not any(c.isdigit() for c in pw):
        return False
    if not any(c in '!@#$%^' for c in pw):
        return False
    return True

tests = ["short", "nouppercase123!", "LongButNoSymbol12", "G00d!Password#"]
for t in tests:
    print(f"{t!r}: {check_password(t)}")

# Hash comparison
h1 = hashlib.sha256(b"sunshine").hexdigest()
h2 = hashlib.sha256(b"Sunshine").hexdigest()
print("\nsunshine hash:", h1)
print("Sunshine hash:", h2)
print("Observation: one capital letter produces a completely different 256-bit hash.")

## Cyber Squad Python Exercises - Randomness & Strong Passwords
Why `random` is predictable, why `secrets` is safe, and how to build a strong password generator. Run each cell in order; complete the STUDENT EXERCISE cells yourself.

---
# MODULE 4: Pseudorandomness

Randomness is critical in cryptography. Weak randomness has caused real-world
catastrophic failures (Android Bitcoin wallet bug, Debian OpenSSL key generation flaw).

| Source | Predictable? | Use for crypto? |
|--------|-------------|----------------|
| `random` module | YES (with seed) | Never |
| `random.SystemRandom` | No (OS entropy) | Only if needed |
| `secrets` module | No (OS entropy) | Yes -- preferred |
| `os.urandom()` | No (OS entropy) | Yes |


In [ ]:
import random, secrets, string, os, time
print('Module 4 ready')


## 4.1 Python random Module: Basic Operations


In [ ]:
import random

# Basic operations
print('randint(1, 100):', random.randint(1, 100))
print('choice([a,b,c]):', random.choice(['alpha', 'beta', 'gamma']))
print('uniform(0, 1)  :', random.uniform(0, 1))

data = list(range(10))
random.shuffle(data)
print('After shuffle  :', data)


## 4.2 Seed Reproducibility Demo

When you set a seed, the sequence is **completely reproducible**.
This is useful for testing but **catastrophic** if used for keys or tokens.


In [ ]:
import random

print('Run 1 with seed 42:')
random.seed(42)
seq1 = [random.randint(0, 100) for _ in range(5)]
print(' ', seq1)

print('Run 2 with seed 42 (identical!):')
random.seed(42)
seq2 = [random.randint(0, 100) for _ in range(5)]
print(' ', seq2)

print(f'Sequences identical: {seq1 == seq2}')  # Expected: True

print('\nNo seed (different each run):')
random.seed(None)
print(' ', [random.randint(0, 100) for _ in range(5)])


## 4.3 /dev/random vs /dev/urandom

On Linux/macOS, the OS provides two random devices:

- `/dev/random`: blocks until enough entropy is gathered from hardware events.
  Very slow. Historically more conservative but now equivalent on modern kernels.
- `/dev/urandom`: non-blocking, uses a CSPRNG seeded from OS entropy.
  Fast and cryptographically secure for all practical purposes.

Python's `os.urandom()` calls `/dev/urandom` (or the Windows equivalent).


In [ ]:
import os

# os.urandom -- cryptographically secure, non-blocking
random_bytes = os.urandom(16)
print(f'os.urandom(16) hex: {random_bytes.hex()}')
print(f'os.urandom(16) int: {int.from_bytes(random_bytes, "big")}')

# Reading /dev/urandom directly (Unix only)
import platform
if platform.system() != 'Windows':
    with open('/dev/urandom', 'rb') as f:
        dev_bytes = f.read(8)
    print(f'/dev/urandom 8 bytes: {dev_bytes.hex()}')
else:
    print('Windows: /dev/urandom not available, os.urandom() is the equivalent')


## 4.4 SystemRandom: OS-Level Randomness via random API


In [ ]:
import random, time

sr = random.SystemRandom()   # Uses os.urandom() under the hood
print('SystemRandom values (non-reproducible):')
print([sr.randint(0, 999) for _ in range(5)])

# Demonstrate: seeding SystemRandom has NO EFFECT
sr.seed(42)   # silently ignored
run1 = [sr.randint(0, 100) for _ in range(5)]
sr.seed(42)
run2 = [sr.randint(0, 100) for _ in range(5)]
print(f'\nSeed ignored -- runs differ: {run1 != run2}')
print(f'Run 1: {run1}')
print(f'Run 2: {run2}')


## 4.5 Student Challenge: Secure Password Generator

Use the `secrets` module (not `random`) to build a password generator that:
- Produces passwords of at least 16 characters
- Guarantees at least one lowercase, uppercase, digit, and special character
- Is cryptographically secure


In [ ]:
# ============ STUDENT EXERCISE ============
# TODO: Implement a secure random password generator
#
# Requirements:
#   - Use secrets module (NOT random)
#   - Length >= 16 (default)
#   - Must include: lowercase, uppercase, digit, special char
#   - Should reject passwords that don't satisfy all requirements
#     (keep regenerating until all checks pass)
#
# Expected output example: 'G@7kMpX!2nRvLs#8'
# ==========================================

import secrets, string

def generate_secure_password(length: int = 16) -> str:
    """
    Generate a cryptographically secure random password.
    Guarantees at least one char from each required character class.
    """
    # TODO: implement this function
    # Hint:
    #   alphabet = string.ascii_lowercase + string.ascii_uppercase + ...
    #   Use a while loop to keep generating until all requirements met
    #   Use secrets.choice(alphabet) for each character
    pass

# Test it:
for i in range(5):
    pwd = generate_secure_password()
    if pwd:  # remove 'if pwd' once implemented
        print(f'Password {i+1}: {pwd} (length: {len(pwd)})')
